# Azure AI Speech: Text-to-Speech and Speech-to-Text

In this notebook you will learn how to use **Azure AI Speech** to:

1. Convert text into natural-sounding audio (**Text-to-Speech**)
2. Control pronunciation and pacing with **SSML** (Speech Synthesis Markup Language)
3. Transcribe an audio file back into text (**Speech-to-Text**)

---

### Prerequisites: Deploy the Azure Resource

Before running this notebook, you need an **Azure AI Speech** resource. Instead of configuring it manually in the Azure Portal, deploy the included **ARM template** which creates the resource in the correct configuration automatically.

> **Tip:** Open `deploy-speech.parameters.json` to customize the resource name, region, or SKU before deploying. The default SKU is **S0** (paid). Change it to **F0** for the free tier.

**1. Deploy the resource** — run the cell below. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

In [3]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", "deploy-speech.json",
     "--parameters", "@deploy-speech.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

{
  "id": "/subscriptions/77bbf218-df70-4269-955d-4818a0834a63/resourceGroups/Foundry/providers/Microsoft.Resources/deployments/deploy-speech",
  "location": null,
  "name": "deploy-speech",
  "properties": {
    "correlationId": "957fbada-9909-486a-b516-174c206bf718",
    "debugSetting": null,
    "dependencies": [],
    "diagnostics": null,
    "duration": "PT2.8189243S",
    "error": null,
    "extensions": [],
    "mode": "Incremental",
    "onErrorDeployment": null,
    "outputResources": [
      {
        "apiVersion": null,
        "extension": null,
        "id": "/subscriptions/77bbf218-df70-4269-955d-4818a0834a63/resourceGroups/Foundry/providers/Microsoft.CognitiveServices/accounts/ashok-speech-demo-2026",
        "identifiers": null,
        "resourceGroup": "Foundry",
        "resourceType": "Microsoft.CognitiveServices/accounts"
      }
    ],
    "outputs": {
      "resourceId": {
        "type": "String",
        "value": "/subscriptions/77bbf218-df70-4269-955d-4818a0834

**2. Assign the required RBAC role** so your identity can use the resource. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a 401 error on your first run, wait a moment and try again.

In [4]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the resource ID from the deployment output
scope_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-speech",
     "--query", "properties.outputs.resourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
resource_id = scope_result.stdout.strip()
print(f"Resource ID:  {resource_id}")

# Step C: Assign the RBAC role
role_result = subprocess.run(
    [AZ, "role", "assignment", "create",
     "--assignee", user_id,
     "--role", "Cognitive Services Speech User",
     "--scope", resource_id],
    capture_output=True, text=True, encoding="utf-8"
)
print(role_result.stdout)
if role_result.returncode != 0:
    print("ERROR:", role_result.stderr)

Your User ID: 8996be55-484a-4c07-b5bf-d92e88a7af83
Resource ID:  /subscriptions/77bbf218-df70-4269-955d-4818a0834a63/resourceGroups/foundry/providers/Microsoft.CognitiveServices/accounts/ashok-speech-demo-2026
{
  "condition": null,
  "conditionVersion": null,
  "createdBy": null,
  "createdOn": "2026-10-12T15:24:49.581799+00:00",
  "delegatedManagedIdentityResourceId": null,
  "description": null,
  "id": "/subscriptions/77bbf218-df70-4269-955d-4818a0834a63/resourceGroups/foundry/providers/Microsoft.CognitiveServices/accounts/ashok-speech-demo-2026/providers/Microsoft.Authorization/roleAssignments/2cbb36ae-1252-4d07-9a79-2a24fc5fab09",
  "name": "2cbb36ae-1252-4d07-9a79-2a24fc5fab09",
  "principalId": "8996be55-484a-4c07-b5bf-d92e88a7af83",
  "principalType": "User",
  "resourceGroup": "foundry",
  "roleDefinitionId": "/subscriptions/77bbf218-df70-4269-955d-4818a0834a63/providers/Microsoft.Authorization/roleDefinitions/f2dc8367-1007-4938-bd23-fe263f013447",
  "scope": "/subscriptions/

**3. Update the `.env` file** in this folder with the values from the deployment output:

```
SPEECH_ENDPOINT="https://<your-account-name>.cognitiveservices.azure.com/"
SPEECH_REGION="eastus2"
SPEECH_RESOURCE_ID="/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<account-name>"
```

You can retrieve all three values at once by running the cell below. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

In [5]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-speech",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

{
  "resourceId": {
    "type": "String",
    "value": "/subscriptions/77bbf218-df70-4269-955d-4818a0834a63/resourceGroups/foundry/providers/Microsoft.CognitiveServices/accounts/ashok-speech-demo-2026"
  },
  "speechEndpoint": {
    "type": "String",
    "value": "https://ashok-speech-demo-2026.cognitiveservices.azure.com/"
  },
  "speechRegion": {
    "type": "String",
    "value": "eastus2"
  }
}



### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

> **Note:** The Speech SDK requires the auth token in a special format:
> `aad#<resource-id>#<token>`. This is handled automatically in Step 2.

## Step 1: Install the Speech SDK

In [6]:
%pip install azure-cognitiveservices-speech azure-identity python-dotenv

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------  2.4/2.4 MB 19.1 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 15.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


## Step 2: Load Environment Variables and Obtain a Token

The Speech SDK doesn't accept `DefaultAzureCredential` directly. Instead
we request a token from Entra ID and pass it as an **authorization token**.

The Speech SDK requires the token in a special format: `aad#<resource-id>#<token>`.
This tells the SDK to use Entra ID authentication with the specified resource.

In [14]:
import azure.cognitiveservices.speech as speechsdk

speech_config = speechsdk.SpeechConfig(
    auth_token=f"aad#{speech_resource_id}#{token}",
    region="eastus2",
)

print("SpeechConfig created successfully!")

SpeechConfig created successfully!


In [20]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Find the .env file next to this notebook's working directory
env_path = Path.cwd() / ".env"

if not env_path.is_file():
    raise FileNotFoundError(
        f".env file not found at: {env_path}\n"
        "Create .env in the Speech folder."
    )

# Load the environment variables
load_dotenv(dotenv_path=env_path, override=True)

speech_endpoint = os.getenv("SPEECH_ENDPOINT")
speech_region = os.getenv("SPEECH_REGION")
speech_resource_id = os.getenv("SPEECH_RESOURCE_ID")

print("Environment file:", env_path)
print("Endpoint loaded:", bool(speech_endpoint))
print("Region loaded:", bool(speech_region))
print("Resource ID loaded:", bool(speech_resource_id))

Environment file: c:\Users\Ashok\OneDrive\Documents\Microsoft-Foundry-Source-Code\Microsoft-Foundry-Source-Code\Foundry-Tools\Speech\.env
Endpoint loaded: True
Region loaded: True
Resource ID loaded: True


In [21]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

token = credential.get_token(
    "https://cognitiveservices.azure.com/.default"
).token

print("Microsoft Entra ID token acquired:", bool(token))

Microsoft Entra ID token acquired: True


In [22]:
import azure.cognitiveservices.speech as speechsdk

speech_config = speechsdk.SpeechConfig(
    auth_token=f"aad#{speech_resource_id}#{token}",
    region=speech_region.strip().lower(),
)

print("SpeechConfig created successfully!")

SpeechConfig created successfully!


In [13]:
from importlib.metadata import version

print("Speech SDK:", version("azure-cognitiveservices-speech"))
print("Azure Identity:", version("azure-identity"))
print("Resource ID:", speech_resource_id)
print("Region:", speech_region)
print("Endpoint:", speech_endpoint)

Speech SDK: 1.52.0
Azure Identity: 1.26.0
Resource ID: None
Region: None
Endpoint: None


In [23]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"

print("Current directory:", Path.cwd())
print(".env exists:", env_path.is_file())

if not env_path.is_file():
    raise FileNotFoundError(
        f"Cannot find .env at {env_path}. "
        "Create .env inside Foundry-Tools/Speech."
    )

load_dotenv(env_path, override=True)

speech_endpoint = os.getenv("SPEECH_ENDPOINT")
speech_region = os.getenv("SPEECH_REGION")
speech_resource_id = os.getenv("SPEECH_RESOURCE_ID")

print("Endpoint loaded:", bool(speech_endpoint))
print("Region loaded:", bool(speech_region))
print("Resource ID loaded:", bool(speech_resource_id))

Current directory: c:\Users\Ashok\OneDrive\Documents\Microsoft-Foundry-Source-Code\Microsoft-Foundry-Source-Code\Foundry-Tools\Speech
.env exists: True
Endpoint loaded: True
Region loaded: True
Resource ID loaded: True


In [24]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

token = credential.get_token(
    "https://cognitiveservices.azure.com/.default"
).token

print("Azure authentication successful:", bool(token))

Azure authentication successful: True


In [25]:
import azure.cognitiveservices.speech as speechsdk

speech_config = speechsdk.SpeechConfig(
    auth_token=f"aad#{speech_resource_id}#{token}",
    region=speech_region.strip().lower(),
)

print("SpeechConfig created successfully!")

SpeechConfig created successfully!


In [26]:
import azure.cognitiveservices.speech as speechsdk

speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"

speech_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config,
    audio_config=None
)

result = speech_synthesizer.speak_text_async(
    "Hello! Your Azure Speech service is working."
).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print("SUCCESS: Text-to-speech completed!")
elif result.reason == speechsdk.ResultReason.Canceled:
    details = result.cancellation_details
    print("Speech synthesis failed:", details.reason)
    print("Error details:", details.error_details)
else:
    print("Unexpected result:", result.reason)

SUCCESS: Text-to-speech completed!


In [27]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

token = credential.get_token(
    "https://cognitiveservices.azure.com/.default"
).token

print("Azure authentication successful:", bool(token))

Azure authentication successful: True


In [28]:
import azure.cognitiveservices.speech as speechsdk

speech_config = speechsdk.SpeechConfig(
    auth_token=f"aad#{speech_resource_id}#{token}",
    region=speech_region.strip().lower(),
)

print("SpeechConfig created successfully!")

SpeechConfig created successfully!


In [29]:
import azure.cognitiveservices.speech as speechsdk

speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"

synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config,
    audio_config=None
)

result = synthesizer.speak_text_async(
    "Hello! Your Azure Speech service is working correctly."
).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print("SUCCESS: Speech synthesis completed!")

elif result.reason == speechsdk.ResultReason.Canceled:
    details = result.cancellation_details
    print("Speech synthesis failed:", details.reason)
    print("Error details:", details.error_details)

else:
    print("Unexpected result:", result.reason)

SUCCESS: Speech synthesis completed!


In [30]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import azure.cognitiveservices.speech as speechsdk

load_dotenv(override=True)

speech_endpoint = os.getenv("SPEECH_ENDPOINT")
speech_region = os.getenv("SPEECH_REGION")
speech_resource_id = os.getenv("SPEECH_RESOURCE_ID")

# Validate configuration
if not speech_endpoint or not speech_region or not speech_resource_id:
    raise ValueError(
        "Missing SPEECH_ENDPOINT, SPEECH_REGION, or SPEECH_RESOURCE_ID."
    )

# Normalize region
speech_region = speech_region.strip().lower().replace(" ", "")

# Authenticate with Microsoft Entra ID
credential = DefaultAzureCredential()
token = credential.get_token(
    "https://cognitiveservices.azure.com/.default"
).token

# Configure Azure Speech SDK
speech_config = speechsdk.SpeechConfig(
    auth_token=f"aad#{speech_resource_id}#{token}",
    region=speech_region,
)

print("SpeechConfig created successfully.")
print("Endpoint:", speech_endpoint)
print("Region:", speech_region)

SpeechConfig created successfully.
Endpoint: https://ashok-speech-demo-2026.cognitiveservices.azure.com/
Region: eastus2


## Step 3: Text-to-Speech -- Generate Audio from Text

We feed a paragraph of plain text to the Speech service and it produces
a `.wav` audio file using a neural voice.

In [31]:
# Choose a neural voice
speech_config.speech_synthesis_voice_name = "en-US-JennyMultilingualNeural"

# The text we want to convert into speech
sample_text = (
    "Cloud computing has revolutionized how businesses operate. "
    "Instead of maintaining expensive on-premises servers, companies "
    "can now rent computing power on demand from providers like "
    "Microsoft Azure, scaling up or down as their needs change."
)

# Save the audio to a WAV file
output_wav = "tts_output.wav"
audio_config = speechsdk.audio.AudioConfig(filename=output_wav)

# Create the synthesizer and generate speech
synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config, audio_config=audio_config
)
result = synthesizer.speak_text_async(sample_text).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"Audio saved to {output_wav}")
else:
    print(f"Speech synthesis failed: {result.reason}")

Audio saved to tts_output.wav


## Step 4: SSML -- Fine-Grained Control over Speech

**SSML** (Speech Synthesis Markup Language) lets you add pauses, change
speaking rate, switch voices, and much more. Below we define the SSML
inline and pass it to the synthesizer.

In [32]:
# Define SSML with pauses and emphasis
ssml_content = """
<speak version="1.0" xml:lang="en-US">
    <voice name="en-US-JennyMultilingualNeural">
        Welcome to this amazing course.
        <break strength="medium" />
        This sample shows how to convert text into
        natural-sounding speech using the Speech service in Foundry tools.
        <break />
        Let's get started!
    </voice>
</speak>
"""

# Save to a different file so we can compare
ssml_output_wav = "ssml_output.wav"
ssml_audio_config = speechsdk.audio.AudioConfig(filename=ssml_output_wav)

ssml_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config, audio_config=ssml_audio_config
)
ssml_result = ssml_synthesizer.speak_ssml_async(ssml_content).get()

if ssml_result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"SSML audio saved to {ssml_output_wav}")
else:
    print(f"SSML synthesis failed: {ssml_result.reason}")

SSML audio saved to ssml_output.wav


## Step 5: Speech-to-Text -- Transcribe an Audio File

Now we do the reverse: take the audio file we generated in Step 3 and
transcribe it back into text.

> **Note:** `recognize_once_async()` transcribes the first utterance
> (up to ~15 seconds). For longer audio, use continuous recognition.

In [33]:
# Point the recognizer at the WAV file we created earlier
speech_config.speech_recognition_language = "en-US"
audio_input = speechsdk.audio.AudioConfig(filename="tts_output.wav")

recognizer = speechsdk.SpeechRecognizer(
    speech_config=speech_config, audio_config=audio_input
)

stt_result = recognizer.recognize_once_async().get()

if stt_result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print("Transcription successful!")
    print(f"Text: {stt_result.text}")
else:
    print(f"Transcription failed: {stt_result.reason}")

# Optionally save the transcription to a file
with open("transcription.txt", "w", encoding="utf-8") as f:
    f.write(stt_result.text)
    print("Saved to transcription.txt")

Transcription successful!
Text: Cloud computing has revolutionized how businesses operate. Instead of maintaining expensive on-premises servers, companies can now rent computing power on demand from providers like Microsoft Azure, scaling up or down as their needs change.
Saved to transcription.txt
